# Attribute Analysis: CIC-IDS2017 Feature Discovery

This notebook analyzes CIC-IDS2017 network flow features to identify discriminative attributes per attack type.

**Outputs:**
- `results/attribute_table.csv`: Summary of detection rules per attack type
- `results/feature_statistics.csv`: Benign vs. attack distributions per feature
- `results/feature_importance.csv`: Feature importance rankings (mutual information)
- `results/correlation_heatmap.png`: Feature correlation heatmap
- `results/attribute_analysis.md`: Detailed findings

## Setup

First, ensure the CIC-IDS2017 parquet file is available.

In [ ]:
import subprocess
import sys
from pathlib import Path

# Ensure we're in the repo root for imports
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")
print(f"Python path: {sys.path[0]}")

## Run Attribute Analysis

Load the CIC-IDS2017 parquet and run the analysis script.

In [ ]:
# Run the analysis script
result = subprocess.run(
    [
        sys.executable,
        "scripts/attribute_analysis.py",
        "--input",
        "data/processed/cic-ids2017.parquet",
        "--output",
        "results/",
    ],
    cwd=repo_root,
    capture_output=False,
)

if result.returncode != 0:
    print(f"Analysis script failed with exit code {result.returncode}")
else:
    print("\n✓ Analysis complete!")

## View Results

Load and display the generated reports.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown

results_dir = repo_root / "results"

# Load attribute table
attr_table = pd.read_csv(results_dir / "attribute_table.csv")
print("Attribute Table (Detection Rules per Attack Type)")
print("=" * 80)
display(attr_table)

In [ ]:
# Load feature importance
importance = pd.read_csv(results_dir / "feature_importance.csv")
print("\nFeature Importance (Top 5 per Attack Type)")
print("=" * 80)

for attack in importance["attack_type"].unique():
    attack_imp = importance[importance["attack_type"] == attack].head(5)
    print(f"\n{attack}:")
    for _, row in attack_imp.iterrows():
        print(f"  {row['rank']}. {row['feature']:40s} MI: {row['importance']:.4f}")

In [ ]:
# Display correlation heatmap
heatmap_path = results_dir / "correlation_heatmap.png"
if heatmap_path.exists():
    print("\nFeature Correlation Heatmap")
    print("=" * 80)
    display(Image(str(heatmap_path)))
else:
    print("Correlation heatmap not found.")

In [ ]:
# Load and display feature statistics
feature_stats = pd.read_csv(results_dir / "feature_statistics.csv")
print("\nTop Feature Statistics (by discrimination power)")
print("=" * 80)

top_discriminative = feature_stats.nlargest(20, "discrimination")
display(top_discriminative[["attack_type", "feature", "benign_median", "attack_median", "discrimination"]])

## Markdown Report

Full analysis findings:

In [ ]:
# Load and display markdown report
report_path = results_dir / "attribute_analysis.md"
if report_path.exists():
    with open(report_path) as f:
        report_md = f.read()
    display(Markdown(report_md))
else:
    print("Report not found.")

## Summary

The attribute analysis has identified:

1. **Per-attack-type feature distributions** (median ± IQR)
2. **Discriminative features** (features most effective at distinguishing attacks from benign traffic)
3. **Suggested detection rules** based on decision trees
4. **Tree precision/recall** to evaluate rule effectiveness
5. **Feature importance rankings** using mutual information

These results inform the StreamGuard detector configuration for supervisor presentation.